# Fase 1 · Etapa 1.1 — Inventário e catálogo (`data/catalog.yaml`)

> **Bases de dados usadas:**
> * **Dados do pesquisador:** os 156 CSVs de `data/raw/` (1ª campanha `exp_5/`, 12
>   arquivos; 2ª campanha livre-livre, 9 configurações × 16 repetições), com integridade
>   conferida por `data/raw/MANIFEST.sha256`; arquivos `.task` do NI (sensibilidades).
> * **Referências externas (só metadados de hardware):** especificações do NI 9234
>   (faixa ±5 V, corte AC 0,5 Hz — ni.com), martelo PCB 086C03 (2,25 mV/N, ±2224 N —
>   pcb.com) e acelerômetro PCB 352C33 (±50 g, 5,8 g — datasheet rev. H).
> Nenhum dado de terceiros entra nas medições.

O que este notebook faz: lê o cabeçalho e o conteúdo de cada arquivo, confere estrutura
(canais, unidades, fs, duração, uniformidade do tempo), e gera o catálogo validado por
pydantic a partir de `configs/data_sources.yaml`.

In [1]:
from collections import Counter

import pandas as pd

from dampid.io.catalog import build_catalog, load_catalog, save_catalog

cat = build_catalog("data/raw", "configs/data_sources.yaml")
save_catalog(cat, "data/catalog.yaml")
cat = load_catalog("data/catalog.yaml")  # round-trip: o arquivo salvo é válido
print(f"{cat.n_files} arquivos catalogados -> data/catalog.yaml")

156 arquivos catalogados -> data/catalog.yaml


## 1. Resumo por configuração de ensaio

In [2]:
rows = []
for cfg, es in cat.by_config().items():
    rows.append({
        "config": cfg,
        "camp.": es[0].campaign,
        "CC": es[0].boundary,
        "n": len(es),
        "datas": ", ".join(sorted({e.timestamp[:10] for e in es})),
        "fs [Hz]": sorted({e.fs_hz for e in es}),
        "duração [s]": sorted({round(e.duration_s, 3) for e in es}),
        "canais": " | ".join(f"{c.name} [{c.unit}]" for c in es[0].channels),
        "anomalias": sum(len(e.anomalies) > 0 for e in es),
    })
inv = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_colwidth", 70):
    print(inv.to_string(index=False))
inv.to_csv("docs/reports/fase1/01_inventario_configuracoes.csv", index=False)

                          config  camp.              CC  n      datas   fs [Hz] duração [s]                                                canais  anomalias
       C1_ACO_CC_mcentro_acentro      1 clamped-clamped  1 25/01/2023 [25600.0]       [4.0]               Input 0 [g] | Input 1 [g] | Input 2 [N]          0
   C1_ACO_CC_mforacentro_acentro      1 clamped-clamped  1 25/01/2023 [25600.0]       [4.0]               Input 0 [g] | Input 1 [g] | Input 2 [N]          0
        C1_ACO_CF_mcentro_aponta      1    clamped-free  1 25/01/2023 [25600.0]       [6.0]               Input 0 [g] | Input 1 [g] | Input 2 [N]          0
         C1_ACO_CF_mponta_aponta      1    clamped-free  1 31/12/1903 [25600.0]       [6.0]               Input 0 [g] | Input 1 [g] | Input 2 [N]          0
  C1_ACRILICO_CC_mcentro_acentro      1 clamped-clamped  1 25/01/2023 [25600.0]       [2.0]               Input 0 [g] | Input 1 [g] | Input 2 [N]          0
C1_ACRILICO_CC_mforaponta_aponta      1 clamped-clamped  1

## 2. Conferências estruturais
* todos os arquivos têm 3 canais, com a mesma unidade por posição (g, g, N)?
* fs é igual em todos? as colunas de tempo avançam exatamente `dt`?
* nomes de canal mudam entre arquivos de uma mesma configuração?

In [3]:
units = Counter(tuple(c.unit for c in e.channels) for e in cat.entries)
fsset = Counter(round(e.fs_hz, 6) for e in cat.entries)
names = Counter((e.campaign, tuple(c.name for c in e.channels)) for e in cat.entries)
print("unidades por posição de canal:", dict(units))
print("taxas de amostragem:", dict(fsset))
print("nomes de canais (campanha, nomes): n arquivos")
for k, v in names.items():
    print("  ", k, v)
dur = Counter((e.config_id.split("_m")[0] if e.campaign == 1 else e.config_id, round(e.duration_s, 3))
              for e in cat.entries)
print("\nduração por configuração:")
for k, v in sorted(dur.items()):
    print("  ", k, v)

unidades por posição de canal: {('g', 'g', 'N'): 156}
taxas de amostragem: {25600.0: 156}
nomes de canais (campanha, nomes): n arquivos
   (1, ('Input 0', 'Input 1', 'Input 2')) 12
   (2, ('Acelerômetro 1', 'Acelerometro 2', 'Martelo')) 64
   (2, ('Acelerometro_1', 'Acelerometro 2', 'Martelo')) 16
   (2, ('Acelerometro 1', 'Acelerometro 2', 'Martelo')) 64

duração por configuração:
   ('C1_ACO_CC', 4.0) 2
   ('C1_ACO_CF', 6.0) 2
   ('C1_ACRILICO_CC', 2.0) 2
   ('C1_ACRILICO_CF', 4.0) 2
   ('C1_AL_CC', 3.0) 2
   ('C1_AL_CF', 4.0) 2
   ('LL_ACO', 5.0) 16
   ('LL_ACRILICO', 5.0) 16
   ('LL_AL', 5.0) 16
   ('LL_FIB_CARBONO', 5.0) 16
   ('LL_FIB_VIDRO_0', 5.0) 16
   ('LL_FIB_VIDRO_90', 5.0) 16
   ('LL_PLA_CAVERNA', 5.0) 16
   ('LL_PLA_LONGARINA_0', 3.0) 15
   ('LL_PLA_LONGARINA_0', 5.0) 1
   ('LL_PLA_LONGARINA_90', 3.0) 15
   ('LL_PLA_LONGARINA_90', 5.0) 1


In [4]:
print("Arquivos com anomalias de estrutura:")
for e in cat.entries:
    for a in e.anomalies:
        print(f"  {e.path}: {a}")

Arquivos com anomalias de estrutura:
  exp_5/Al_engastado_livre_mponta_aponta.csv: channel 0: time break after sample 102400 (t=3.99996 -> 2.75574e+14); 2193 trailing rows discarded
  exp_5/Al_engastado_livre_mponta_aponta.csv: channel 1: time break after sample 102400 (t=3.99996 -> 0.00322562); 2193 trailing rows discarded
  exp_5/Al_engastado_livre_mponta_aponta.csv: channel 2: time break after sample 102400 (t=3.99996 -> 0.0773144); 2193 trailing rows discarded


## 3. Tabela completa por arquivo (exportada para o relatório)

In [5]:
full = pd.DataFrame([{
    "id": e.id, "arquivo": e.path, "campanha": e.campaign, "config": e.config_id,
    "espécime": e.specimen, "CC": e.boundary, "rep.": e.repetition, "data/hora": e.timestamp,
    "fs [Hz]": e.fs_hz, "amostras": e.n_samples, "duração [s]": round(e.duration_s, 4),
    "martelo (nome do arquivo)": e.hammer_position, "acel. (nome do arquivo)": e.accel_position,
    "anomalias": "; ".join(e.anomalies),
} for e in cat.entries])
full.to_csv("docs/reports/fase1/01_catalogo_arquivos.csv", index=False)
print(full.groupby(["campanha", "config"]).size().to_string())

campanha  config                          
1         C1_ACO_CC_mcentro_acentro            1
          C1_ACO_CC_mforacentro_acentro        1
          C1_ACO_CF_mcentro_aponta             1
          C1_ACO_CF_mponta_aponta              1
          C1_ACRILICO_CC_mcentro_acentro       1
          C1_ACRILICO_CC_mforaponta_aponta     1
          C1_ACRILICO_CF_mcentro_aponta        1
          C1_ACRILICO_CF_mponta_aponta         1
          C1_AL_CC_mcentro_acentro             1
          C1_AL_CC_mforacentro_acentro         1
          C1_AL_CF_mcentro_aponta              1
          C1_AL_CF_mponta_aponta               1
2         LL_ACO                              16
          LL_ACRILICO                         16
          LL_AL                               16
          LL_FIB_CARBONO                      16
          LL_FIB_VIDRO_0                      16
          LL_FIB_VIDRO_90                     16
          LL_PLA_CAVERNA                      16
          LL_PLA_LONGARINA

## 4. Campos que continuam `TODO` (só o pesquisador tem)
Posição e direção de cada acelerômetro, ponto de impacto, ponteira do martelo, cotas de
fibra de carbono, espessura do tubo de fibra de vidro, cotas e parâmetros de impressão
do PLA, ligas do aço e do alumínio, e o que diferencia `PLA_LONGARINA` de
`PLA_LONGARINA_90`. Ficam como `null` no catálogo — nunca são preenchidos por palpite.

In [6]:
todo = {
    "location (posição/direção do sensor)": sum(c.location is None for e in cat.entries for c in e.channels if c.role == "accelerometer"),
    "impact_location": sum(e.impact_location is None for e in cat.entries),
    "hammer_tip": sum(e.hammer_tip is None for e in cat.entries),
}
print(todo)

{'location (posição/direção do sensor)': 300, 'impact_location': 156, 'hammer_tip': 156}
